# tSNE plotting of chimeara + atlas integration

In [ ]:
suppressPackageStartupMessages({
    library(dplyr)
    library(ggplot2)
    library(Matrix)
    library(scran)
    library(Rtsne)
    library(BiocParallel)
    require(irlba)
    library(batchelor)
    library(stringr)
    library(gridExtra)
    library(M3C)
})

In [ ]:
# Load in data
out_folder = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/data/"
big_meta = read.table(paste0(out_folder,"big_meta.tab"), header = TRUE, sep = "\t", stringsAsFactors = FALSE, comment.char = "$")
pca_before = as.data.frame(readRDS(paste0(out_folder, 'uncorrected_PCs.rds')))
pca_before$cell <- rownames(pca_before)
pca_after = as.data.frame(read.csv(paste0(out_folder, 'corrected_pc_complete.csv'), header=T, row.names=1,sep=","))
pca_after$cell <- rownames(pca_after)

In [ ]:
# add metadata to PCs
pca_before <- merge(pca_before, big_meta, by='cell')
rownames(pca_before) <- pca_before$cell
pca_after <- merge(pca_after, big_meta, by='cell')
rownames(pca_after) <- pca_after$cell

In [ ]:
# calculate tsne's on whole object and chimeara seperately
### chimaera only
# before correction
pca_before$origin <- sapply(strsplit(pca_before$cell,"_"), `[`, 1)
chim_before <- pca_before %>% filter(origin=='chim') %>% select(2:31) # use first 30 PCs
chim_before_tsne = Rtsne(chim_before, pca = FALSE)$Y
chim_before_tsne <- as.data.frame(chim_before_tsne)
colnames(chim_before_tsne) <- c('tsne1','tsne2')
chim_before_tsne$cell <- rownames(chim_before)
chim_before_tsne <- merge(chim_before_tsne, big_meta, by='cell')
# write
write.csv(chim_before_tsne, paste0(out_folder, 'tsne_before_correction_chim.csv')) 
print('finished first tsne')


# after correction
pca_after$origin <- sapply(strsplit(pca_after$cell,"_"), `[`, 1)
chim_after <- pca_after %>% filter(origin=='chim') %>% select(2:31) # use first 30 PCs
chim_after_tsne = Rtsne(chim_after, pca = FALSE)$Y
chim_after_tsne<- as.data.frame(chim_after_tsne)
colnames(chim_after_tsne) <- c('tsne1','tsne2')
chim_after_tsne$cell <- rownames(chim_after)
chim_after_tsne <- merge(chim_after_tsne, big_meta, by='cell')
# write
write.csv(chim_after_tsne, paste0(out_folder, 'tsne_after_correction_chim.csv')) 
print('finished second tsne')

### whole object
# before correction
pca_before$origin <- sapply(strsplit(pca_before$cell,"_"), `[`, 1)
pca_before <- pca_before %>% select(2:31) # use first 30 PCs
pca_before_tsne = Rtsne(pca_before, pca = FALSE)$Y
pca_before_tsne <- as.data.frame(pca_before_tsne)
colnames(pca_before_tsne) <- c('tsne1','tsne2')
pca_before_tsne$cell <- rownames(pca_before)
pca_before_tsne <- merge(pca_before_tsne, big_meta, by='cell')
# write
write.csv(pca_before_tsne, paste0(out_folder, 'tsne_before_correction.csv')) 
print('finished third tsne')

# after correction
pca_after$origin <- sapply(strsplit(pca_after$cell,"_"), `[`, 1)
pca_after <- pca_after %>% select(2:31) # use first 30 PCs
pca_after_tsne = Rtsne(pca_after, pca = FALSE)$Y
pca_after_tsne <- as.data.frame(pca_after_tsne)
colnames(pca_after_tsne) <- c('tsne1','tsne2')
pca_after_tsne$cell <- rownames(pca_after)
pca_after_tsne <- merge(pca_after_tsne, big_meta, by='cell')
# write
write.csv(pca_after_tsne, paste0(out_folder, 'tsne_after_correction.csv')) 
print('finished fourth tsne')

In [ ]:
# load in data
out_folder = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/data/"
chim_before_tsne = as.data.frame(read.csv(paste0(out_folder, 'tsne_before_correction_chim.csv'), header=T, row.names=1,sep=","))
chim_after_tsne = as.data.frame(read.csv(paste0(out_folder, 'tsne_after_correction_chim.csv'), header=T, row.names=1,sep=","))
pca_before_tsne = as.data.frame(read.csv(paste0(out_folder, 'tsne_before_correction.csv'), header=T, row.names=1,sep=","))
pca_after_tsne = as.data.frame(read.csv(paste0(out_folder, 'tsne_after_correction.csv'), header=T, row.names=1,sep=","))

In [ ]:
# plot tsnes
# chimaera
options(repr.plot.width = 10, repr.plot.height = 5)
p1 <- ggplot(chim_before_tsne, aes(tsne1, tsne2, color=sample)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('chimaera only, before correction, color = samples')
p2 <- ggplot(chim_before_tsne, aes(tsne1, tsne2, color=tdTom)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('chimaera only, before correction, color =  tomato')

p3 <- ggplot(chim_after_tsne, aes(tsne1, tsne2, color=sample)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('chimaera only, after correction, color = samples')
p4 <- ggplot(chim_after_tsne, aes(tsne1, tsne2, color=tdTom)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('chimaera only, after correction, color = tomato')

# whole object
p5 <- ggplot(pca_before_tsne, aes(tsne1, tsne2, color=sample)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('whole object before correction, color = sample')
p6 <- ggplot(pca_after_tsne, aes(tsne1, tsne2, color=sample)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('whole object after correction, color = sample')

p7 <- ggplot(pca_before_tsne, aes(tsne1, tsne2, color=tdTom)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('whole object before correction, color = tomato')
p8 <- ggplot(pca_after_tsne, aes(tsne1, tsne2, color=tdTom)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('whole object after correction, color = tomato')

p9 <- ggplot(pca_before_tsne, aes(tsne1, tsne2, color=stage)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('whole object before correction, color = timepoint')
p10 <- ggplot(pca_after_tsne, aes(tsne1, tsne2, color=stage)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('whole object after correction, color = timepoint')

atlas_before_tsne <- pca_before_tsne %>% filter(!is.na(celltype))
p11 <- ggplot(atlas_before_tsne, aes(tsne1, tsne2, color=celltype)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('atlas only before correction, color = celltype')
atlas_after_tsne <- pca_after_tsne %>% filter(!is.na(celltype))
p12 <- ggplot(atlas_after_tsne, aes(tsne1, tsne2, color=celltype)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('atlas only after correction, color = celltype')

pca_before_85 <- pca_before_tsne %>% filter(stage == 'E8.5')
p13 <- ggplot(pca_before_85, aes(tsne1, tsne2, color=tdTom)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('E8.5 before correction, color = tomato')

pca_after_85 <- pca_after_tsne %>% filter(stage == 'E8.5')
p14 <- ggplot(pca_after_85, aes(tsne1, tsne2, color=tdTom)) +
    geom_point(alpha=0.4, size = 0.4) +
    theme_classic() +
    ggtitle('E8.5 after correction, color = tomato')

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 30)
    grid.arrange(p1, p3, p2, p4, p5, p6, p7, p8, p9, p10, p11, p12, nrow =6)

In [ ]:
pdf(paste0(plot_folder, "batch_correction_comparisons_chimaera.pdf"), useDingbats = FALSE, height=10, width=10)
    grid.arrange(p1, p3, p2, p4, nrow =2)
dev.off()

In [ ]:
# plot to pdf --> seperate pdfs bc they are so insanely huge it's hard to load in 
plot_folder = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/plots/"
pdf(paste0(plot_folder, "batch_correction_comparisons.pdf"), useDingbats = FALSE, height=40, width=30)
    grid.arrange(p1, p3, p2, p4, p5, p6, p7, p8, p9, p10, p11, p12, nrow =6)
dev.off()

pdf(paste0(plot_folder, "batch_correction_comparisons_chimaera.pdf"), useDingbats = FALSE, height=10, width=10)
    grid.arrange(p1, p3, p2, p4, nrow =2)
dev.off()

pdf(paste0(plot_folder, "batch_correction_comparisons_sample.pdf"), useDingbats = FALSE, height=8, width=25)
    grid.arrange(p5, p6, nrow =1)
dev.off()

pdf(paste0(plot_folder, "batch_correction_comparisons_tomato.pdf"), useDingbats = FALSE, height=8, width=25)
    grid.arrange(p7, p8, nrow =1)
dev.off()

pdf(paste0(plot_folder, "batch_correction_comparisons_timepoint.pdf"), useDingbats = FALSE, height=8, width=25)
    grid.arrange(p9, p10, nrow =1)
dev.off()

pdf(paste0(plot_folder, "batch_correction_comparisons_celltype.pdf"), useDingbats = FALSE, height=8, width=25)
    grid.arrange(p11, p12, nrow =1)
dev.off()

pdf(paste0(plot_folder, "batch_correction_comparisons_E85.pdf"), useDingbats = FALSE, height=8, width=25)
    grid.arrange(p11, p12, nrow =1)
dev.off()

pdf(paste0(plot_folder, "batch_corrected_plots.pdf"), useDingbats = FALSE, height=15, width=25)
    grid.arrange(p6, p8, p12, p10, nrow =2)
dev.off()

In [ ]:
# plot to png to stop laptop from burning
plot_folder = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/plots/"
png(paste0(plot_folder, "batch_correction_comparisons.png"), height=5000, width=4500)
    grid.arrange(p1, p3, p2, p4, p5, p6, p7, p8, p9, p10, p11, p12, p13, p14, nrow =7)
dev.off()

png(paste0(plot_folder, "batch_correction_comparisons_chimaera.png"), height=1000, width=1000)
    grid.arrange(p1, p3, p2, p4, nrow =2)
dev.off()

png(paste0(plot_folder, "batch_correction_comparisons_sample.png"), height=1000, width=2500)
    grid.arrange(p5, p6, nrow =1)
dev.off()

png(paste0(plot_folder, "batch_correction_comparisons_tomato.png"), height=1000, width=2500)
    grid.arrange(p7, p8, nrow =1)
dev.off()

png(paste0(plot_folder, "batch_correction_comparisons_timepoint.png"), height=1000, width=2500)
    grid.arrange(p9, p10, nrow =1)
dev.off()

png(paste0(plot_folder, "batch_correction_comparisons_celltype.png"), height=1000, width=2500)
    grid.arrange(p11, p12, nrow =1)
dev.off()

png(paste0(plot_folder, "batch_correction_comparisons_E85.png"), height=1000, width=2500)
    grid.arrange(p13, p14, nrow =1)
dev.off()

png(paste0(plot_folder, "batch_corrected_plots.png"), height=2000, width=3500)
    grid.arrange(p6, p8, p12, p10, nrow =2)
dev.off()